# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Dataset published: {metadata.datePublished}")
print(f"Keywords: {metadata.keywords}")
print(f"Version: {metadata.version}")

## 2. Data Overview
Review available record sets, fields, their IDs, and contents. All identifiers are referenced using their `@id` fields.

In [ ]:
# List all record sets in the dataset and their fields/columns

record_sets = []
for rs in dataset.iter_record_sets():
    # Each record set has '@id', 'name', and may contain 'fields'
    print(f"RecordSet @id: {rs['@id']}, name: {rs['name']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for field in fields:
        if isinstance(field, dict):
            print(f"    Field @id: {field.get('@id', None)}, name: {field.get('name', None)} (type: {field.get('dataType', None)})")
        else:
            print(f"    Field @id: {field}")
    record_sets.append(rs['@id'])
    print()

print(f"Total record sets: {len(record_sets)}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview. All entities are referenced by their `@id`.

In [ ]:
import collections
# For this exploration, we will use the first record set (update as needed if you want another one)
if len(record_sets) == 0:
    raise RuntimeError("No record sets found in metadata.")
record_set_id = record_sets[0]

dataframes = {}
records = list(dataset.records(record_set=record_set_id))
dataframes[record_set_id] = pd.DataFrame(records)
print(f"Extracted columns (@id or field name) from {record_set_id}:")
print(dataframes[record_set_id].columns.tolist())

dataframes[record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations demonstrated include removing outliers, transforming data distributions, and grouping data by key attributes.

Replace references with field `@id`s. We'll demonstrate with the GAD-7 Score field if present.

In [ ]:
# Let's inspect all numeric columns and choose a field for analysis
df = dataframes[record_set_id]
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
print(f"Numeric fields detected: {numeric_cols}")

if numeric_cols:
    # Select the first numeric field for example (likely to be GAD-7, PHQ-9, or PSQ total)
    numeric_field = numeric_cols[0]
else:
    raise RuntimeError("No numeric fields found for EDA.")

# Set a threshold to filter high scores, e.g., GAD-7 > 9 (moderate/severe anxiety)
threshold = 9
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

group_field_candidates = ['level_of_education', 'gender', 'village', 'marital_status']
group_field = None
for candidate in group_field_candidates:
    if candidate in df.columns:
        group_field = candidate
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"Mean_{numeric_field}")
    print(f"Grouped data by {group_field} (showing mean):")
    print(grouped_df.head())
else:
    print("No suitable group field found for aggregation.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll plot the distribution of the selected numeric field and its mean by group if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the entire numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field], kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If grouped, show group means
if group_field and group_field in df.columns:
    group_means = df.groupby(group_field)[numeric_field].mean().sort_values()
    plt.figure(figsize=(10, 5))
    group_means.plot(kind='bar')
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.show()

## 6. Conclusion
Through this notebook, we've loaded and explored a FAIR² mental health dataset from Kilifi County, Kenya, using the `mlcroissant` library.

- We inspected dataset metadata and structure using `@id` references.
- We extracted and processed survey data, filtered and normalized scores, and explored group differences.
- Visualizations provide insight into numeric field distributions and potential demographic patterns.

The Croissant standard facilitates reproducible, machine-actionable data analysis for open science and health research.